In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.market_math import compute_log_spread
from statsmodels.tsa.stattools import adfuller


# 10 Frozen Equilibrium Shift Diagnostics

Compare frozen equilibrium parameters with trailing realized spreads. These are diagnostics, not a rolling-recalibration strategy or a causal explanation of drawdown.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Reconstruct frozen spreads

Use the same alpha, beta and equilibrium as Module 07.


In [ ]:
train = pd.read_parquet("train_prices.parquet")
test = pd.read_parquet("test_prices.parquet")
full = pd.concat([train, test])
params = pd.read_parquet("pair_parameters.parquet")
rows = []
rolling_spreads = {}
for row in params.itertuples():
    spread = compute_log_spread(full, row.dependent, row.independent, row.alpha, row.beta)
    shift = (spread.rolling(63).mean() - row.mu) / np.sqrt(row.variance)
    rolling_spreads[row.pair] = shift.reindex(test.index)
    recent = spread.iloc[-252:]
    try:
        ordinary_adf_p = float(adfuller(recent, regression="c", autolag="AIC")[1])
    except ValueError:
        ordinary_adf_p = np.nan
    rows.append(
        dict(
            pair=row.pair,
            mean_absolute_shift_z=shift.reindex(test.index).abs().mean(),
            final_shift_z=shift.iloc[-1],
            trailing_ordinary_adf_p=ordinary_adf_p,
        )
    )
metrics = pd.DataFrame(rows).sort_values("mean_absolute_shift_z", ascending=False)
metrics.to_parquet("equilibrium_shift_metrics.parquet")
pd.DataFrame(rolling_spreads).to_parquet("rolling_equilibrium_shift.parquet")
display(metrics.head(15))


## 3. Visual inspection

Trailing ordinary ADF p-values here are residual diagnostics, not the formal formation Engle–Granger test. Large displacement is descriptive evidence and does not identify a unique drawdown mechanism.


In [ ]:
if not metrics.empty:
    examples = metrics.pair.head(4).tolist()
    pd.DataFrame(rolling_spreads)[examples].plot(
        figsize=(11, 4),
        title="Trailing 63-session mean displacement in frozen standard deviations",
    )
    plt.axhline(0, color="black", linestyle="--")
    plt.show()
